## FoodCNN — CNN Training Phase 2: Full Fine-Tuning

Load the Phase 1 checkpoint, **unfreeze the entire ResNet-50 backbone**,
and fine-tune end-to-end with a much lower learning rate.
Differential LRs are used: backbone gets 10x lower LR than the projection head.


#### Imports

In [ ]:
import os
import json
import torch
import torch.nn as nn
import matplotlib.pyplot as plt
from torchvision import transforms
from torchvision.models import resnet50, ResNet50_Weights
from torch.utils.data import Dataset, DataLoader
from PIL import Image


#### Config

In [ ]:
BASE_PATH   = "datasets/food-101"
BATCH_SIZE  = 32
NUM_EPOCHS  = 10      # phase 2 fine-tunes the full network
LR_HEAD     = 1e-4    # projection head LR
LR_BACKBONE = 1e-5    # backbone LR (10x lower to preserve pretrained features)
NUM_WORKERS = 2
CKPT_DIR    = "checkpoints"
os.makedirs(CKPT_DIR, exist_ok=True)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)


#### Dataset & Transforms

In [ ]:
class Food101Dataset(Dataset):
    def __init__(self, base_path, split_file, transform=None):
        self.image_path = os.path.join(base_path, "images")
        self.transform  = transform
        with open(os.path.join(base_path, "meta", split_file)) as f:
            lines = f.read().splitlines()
        self.image_paths = [os.path.join(self.image_path, l + ".jpg") for l in lines]
        raw_labels       = [l.split("/")[0] for l in lines]
        self.classes     = sorted(set(raw_labels))
        c2i              = {c: i for i, c in enumerate(self.classes)}
        self.labels      = [c2i[l] for l in raw_labels]

    def __len__(self):  return len(self.image_paths)

    def __getitem__(self, idx):
        img = Image.open(self.image_paths[idx]).convert("RGB")
        if self.transform: img = self.transform(img)
        return img, self.labels[idx]


In [ ]:
train_transform = transforms.Compose([
    transforms.RandomResizedCrop(224),
    transforms.RandomHorizontalFlip(),
    transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2, hue=0.1),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])

val_transform = transforms.Compose([
    transforms.Resize(256),
    transforms.CenterCrop(224),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])


In [ ]:
train_full = Food101Dataset(BASE_PATH, "train.txt", train_transform)
train_size = int(0.8 * len(train_full))
val_size   = len(train_full) - train_size
train_subset, val_subset_raw = torch.utils.data.random_split(train_full, [train_size, val_size])

class TransformSubset(Dataset):
    def __init__(self, subset, transform):
        self.subset    = subset
        self.transform = transform
    def __len__(self): return len(self.subset)
    def __getitem__(self, idx):
        img_path = self.subset.dataset.image_paths[self.subset.indices[idx]]
        label    = self.subset.dataset.labels[self.subset.indices[idx]]
        img = Image.open(img_path).convert("RGB")
        if self.transform: img = self.transform(img)
        return img, label

val_dataset  = TransformSubset(val_subset_raw, val_transform)
test_dataset = Food101Dataset(BASE_PATH, "test.txt", val_transform)

train_loader = DataLoader(train_subset,  batch_size=BATCH_SIZE, shuffle=True,  num_workers=NUM_WORKERS)
val_loader   = DataLoader(val_dataset,   batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS)
test_loader  = DataLoader(test_dataset,  batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS)

print(f"Train: {len(train_subset)}  Val: {len(val_dataset)}  Test: {len(test_dataset)}")


#### Load Phase 1 Checkpoint & Unfreeze Backbone

We start from the best Phase 1 weights so the projection head is already warm.
Then we unfreeze all layers and use differential learning rates:
- Backbone: `1e-5` (small — preserve pretrained features)
- Projection head: `1e-4` (larger — head still needs more adjustment)


In [ ]:
# Rebuild model architecture (must match Phase 1)
backbone = resnet50(weights=ResNet50_Weights.IMAGENET1K_V1)
backbone.fc = nn.Sequential(
    nn.Linear(2048, 512),
    nn.ReLU(),
    nn.Dropout(0.3),
    nn.Linear(512, 101),
)

# Load Phase 1 weights
ckpt = torch.load(os.path.join(CKPT_DIR, "phase1_best.pth"), map_location=device)
backbone.load_state_dict(ckpt["model_state"])
print(f"Loaded phase1_best.pth  epoch={ckpt['epoch']}  val_acc={ckpt['val_acc']:.4f}")

# Unfreeze ALL parameters for end-to-end fine-tuning
for param in backbone.parameters():
    param.requires_grad = True

backbone = backbone.to(device)

trainable = sum(p.numel() for p in backbone.parameters() if p.requires_grad)
total     = sum(p.numel() for p in backbone.parameters())
print(f"Trainable params: {trainable:,} / {total:,}")


#### Training Loop — Phase 2 (Differential LRs)

In [ ]:
# Differential learning rates: backbone gets 10x lower LR than head
backbone_params = [p for n, p in backbone.named_parameters() if "fc" not in n]
head_params     = list(backbone.fc.parameters())

optimizer = torch.optim.Adam([
    {"params": backbone_params, "lr": LR_BACKBONE},
    {"params": head_params,     "lr": LR_HEAD},
])
criterion = nn.CrossEntropyLoss()
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=NUM_EPOCHS)

best_val_acc = ckpt["val_acc"]   # start from phase-1 best
history      = {"train_loss": [], "val_acc": []}

for epoch in range(NUM_EPOCHS):
    backbone.train()
    total_loss  = 0.0
    num_batches = len(train_loader)

    for batch_idx, (images, labels) in enumerate(train_loader):
        images, labels = images.to(device), labels.to(device)
        optimizer.zero_grad()
        loss = criterion(backbone(images), labels)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
        if (batch_idx + 1) % 100 == 0:
            print(f"  [{epoch+1}/{NUM_EPOCHS}] batch {batch_idx+1}/{num_batches}"
                  f"  loss={loss.item():.4f}")

    scheduler.step()

    # Validation
    backbone.eval()
    correct, total = 0, 0
    with torch.no_grad():
        for images, labels in val_loader:
            images, labels = images.to(device), labels.to(device)
            _, preds = torch.max(backbone(images), 1)
            correct += (preds == labels).sum().item()
            total   += labels.size(0)

    val_acc  = correct / total
    avg_loss = total_loss / num_batches
    history["train_loss"].append(avg_loss)
    history["val_acc"].append(val_acc)
    print(f"Epoch [{epoch+1}/{NUM_EPOCHS}]  loss={avg_loss:.4f}  val_acc={val_acc:.4f}"
          f"  lr_backbone={scheduler.get_last_lr()[0]:.2e}")

    if val_acc > best_val_acc:
        best_val_acc = val_acc
        torch.save({
            "epoch":           epoch + 1,
            "model_state":     backbone.state_dict(),
            "optimizer_state": optimizer.state_dict(),
            "val_acc":         val_acc,
        }, os.path.join(CKPT_DIR, "phase2_best.pth"))
        print(f"  -> Saved phase2_best.pth (val_acc={val_acc:.4f})")

print(f"Phase 2 complete. Best val acc: {best_val_acc:.4f}")


#### Final Evaluation & Export

Load the best Phase 2 checkpoint, run on the test set, plot the training curve,
and export the visual embedding backbone for use in downstream fusion.


In [ ]:
# Load best phase-2 checkpoint
ckpt2 = torch.load(os.path.join(CKPT_DIR, "phase2_best.pth"), map_location=device)
backbone.load_state_dict(ckpt2["model_state"])
print(f"Loaded phase2_best.pth  epoch={ckpt2['epoch']}  val_acc={ckpt2['val_acc']:.4f}")

# Test accuracy
backbone.eval()
correct, total = 0, 0
with torch.no_grad():
    for images, labels in test_loader:
        images, labels = images.to(device), labels.to(device)
        _, preds = torch.max(backbone(images), 1)
        correct += (preds == labels).sum().item()
        total   += labels.size(0)
print(f"Phase 2 Test Accuracy: {correct/total:.4f}")

# Training curve
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].plot(history["train_loss"], marker="o")
axes[0].set_title("Phase 2 — Training Loss")
axes[0].set_xlabel("Epoch"); axes[0].set_ylabel("Loss"); axes[0].grid(True)
axes[1].plot(history["val_acc"], marker="o", color="orange")
axes[1].set_title("Phase 2 — Validation Accuracy")
axes[1].set_xlabel("Epoch"); axes[1].set_ylabel("Accuracy"); axes[1].grid(True)
plt.tight_layout()
plt.savefig(os.path.join(CKPT_DIR, "phase2_curve.png"), dpi=150)
plt.show()
print("Saved phase2_curve.png")


In [ ]:
# Export the visual embedding backbone (without the classifier head)
# This produces the 512-d visual embed v used in downstream multimodal fusion.
import copy

visual_encoder = copy.deepcopy(backbone)

# Replace classifier with Identity so forward() returns the 512-d embed
visual_encoder.fc = nn.Sequential(
    nn.Linear(2048, 512),
    nn.ReLU(),
)  # drop Dropout + final Linear — we want the raw 512-d embedding

# Copy weights from trained backbone.fc[0] and fc[1]
visual_encoder.fc[0].weight.data = backbone.fc[0].weight.data.clone()
visual_encoder.fc[0].bias.data   = backbone.fc[0].bias.data.clone()

torch.save(visual_encoder.state_dict(),
           os.path.join(CKPT_DIR, "visual_encoder.pth"))
print("Visual encoder (512-d) saved to checkpoints/visual_encoder.pth")

# Quick sanity check: verify output shape is (batch, 512)
visual_encoder = visual_encoder.to(device).eval()
dummy = torch.randn(4, 3, 224, 224).to(device)
with torch.no_grad():
    embed = visual_encoder(dummy)
print(f"Visual embed shape: {embed.shape}  (expected: [4, 512])")
